# 02 — Vocal2BGM: gerar arranjo a partir da sua voz

Este notebook usa o **ACE-Step 1.5** para gerar um arranjo musical completo  
ao redor da sua voz isolada.

**O que acontece com sua voz:**  
Sua voz real é codificada em espaço latente, o ACE-Step gera o acompanhamento  
ao redor dela, e a voz reconstruída aparece na saída. Não é um sample direto —  
é uma reconstrução via autoencoder. Em gravações limpas, a diferença é mínima.

**Entrada:** `audio/stems/htdemucs/<nome>/vocals.wav` (do notebook 01)  
**Saída:** `audio/output/<nome>_bgm.wav`

---
**Parâmetros principais:**
| Parâmetro | Faixa | Efeito |
|---|---|---|
| `guidance_scale` | 4–10 | adesão ao prompt: maior = mais fiel |
| `num_steps` | 8–50 | qualidade: mais passos = melhor (mais lento) |

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from scripts.utils import (
    carregar_audio, salvar_audio, plotar_audio,
    log_sessao, verificar_gpu,
    AUDIO_STEMS, AUDIO_OUT
)

device = verificar_gpu()

In [ ]:
# ── CONFIGURAÇÃO — edite aqui ─────────────────────────────────────────────────

# Caminho da voz isolada (saída do notebook 01)
VOCAL_PATH = AUDIO_STEMS / 'htdemucs' / 'minha_musica' / 'vocals.wav'

# Descrição do arranjo desejado
# Seja específico: gênero, instrumentos, mood, andamento, caráter da voz
PROMPT = (
    'samba brasileiro, violão clássico, cavaquinho, pandeiro, surdo, '
    'baixo acústico, voz masculina, alegre, andamento moderado, produção limpa'
)

# Parâmetros de geração
GUIDANCE_SCALE = 7.5   # 4.0–10.0  (quanto mais alto, mais fiel ao prompt)
NUM_STEPS      = 27    # 8–50      (mais passos = melhor qualidade, mais lento)

# Nome do arquivo de saída
NOME_SAIDA = 'samba_01_bgm'

# ─────────────────────────────────────────────────────────────────────────────

assert VOCAL_PATH.exists(), f'Voz não encontrada: {VOCAL_PATH}\nExecute primeiro o notebook 01.'
vocal_audio, sr_vocal = carregar_audio(VOCAL_PATH)
plotar_audio(vocal_audio, sr_vocal, titulo='Voz isolada (entrada)')

In [ ]:
print('Carregando ACE-Step 1.5...')
print('(Os pesos (~5 GB) serão baixados na primeira execução)')

from acestep import ACEStep
model = ACEStep.from_pretrained('ACE-Step-1.5')
print('✓ Modelo carregado')

In [ ]:
import time

print(f'Gerando arranjo com guidance={GUIDANCE_SCALE}, steps={NUM_STEPS}...')
print(f'Prompt: {PROMPT}\n')

t0 = time.time()

bgm_audio, sr_out = model.vocal_to_bgm(
    vocal_audio=vocal_audio,
    vocal_sample_rate=sr_vocal,
    style_prompt=PROMPT,
    guidance_scale=GUIDANCE_SCALE,
    num_steps=NUM_STEPS,
)

elapsed = time.time() - t0
print(f'✓ Gerado em {elapsed:.1f}s')

# Salvar
import numpy as np
audio_np = bgm_audio.squeeze().numpy().T
caminho_saida = AUDIO_OUT / f'{NOME_SAIDA}.wav'
salvar_audio(audio_np, sr_out, caminho_saida)

plotar_audio(audio_np, sr_out, titulo=f'Resultado: {NOME_SAIDA}')

In [ ]:
# Reproduzir no notebook
from IPython.display import Audio, display

print('Entrada (voz isolada):')
display(Audio(str(VOCAL_PATH)))

print('\nSaída (música completa):')
display(Audio(str(caminho_saida)))

In [ ]:
# ── REGISTRO DA SESSÃO — edite as observações antes de executar ───────────────

log_sessao(
    titulo=f'Vocal2BGM — {NOME_SAIDA}',
    notas=f"""
- Voz de entrada: {VOCAL_PATH.name}
- Prompt: {PROMPT}
- guidance_scale: {GUIDANCE_SCALE}
- num_steps: {NUM_STEPS}
- Tempo de geração: {elapsed:.1f}s
- Saída: {caminho_saida.name}
- Avaliação: (1–5 estrelas)
- O que ficou bom:
- O que ficou ruim:
- Próximo teste: (ex: aumentar guidance, mudar prompt, trocar modelo)
"""
)

## Experimentos sugeridos

Execute a célula de geração novamente com variações nos parâmetros e compare:

| Teste | guidance | num_steps | Mudança no prompt |
|---|---|---|---|
| 01 | 7.5 | 27 | base |
| 02 | 5.0 | 27 | menos fiel ao prompt, mais criativo |
| 03 | 9.0 | 27 | mais fiel ao prompt |
| 04 | 7.5 | 50 | mais qualidade |
| 05 | 7.5 | 27 | mudar gênero no prompt |

Registre cada experimento com `log_sessao()` para manter histórico.